# Rapport comparatif multi-SLM

Couche d'affichage. Toute la logique vit dans `reporting/report.py`, pour qu'elle reste
testable et rejouable en une commande (`python -m reporting.report`).

Le rapport se construit exclusivement depuis `results/` : aucune inférence n'est relancée.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from IPython.display import Markdown, display

from config import BENCHMARK_CAVEATS, COMPOSITE_WEIGHTS, JUDGE_MODEL, Task
from reporting.report import best_output, build_report, task_table

pd.set_option("display.max_colwidth", 60)
print(f"Juge : {JUDGE_MODEL} | pondérations : {COMPOSITE_WEIGHTS}")

## Tableaux comparatifs, une tâche à la fois

In [ ]:
COLUMNS = [
    "model",
    "judge_score",
    "judge_pass_rate",
    "code_mean",
    "latency_mean_s",
    "tokens_per_second",
    "total_gb",
    "gpu_fraction",
    "n_errors",
    "composite",
]

tables = {task: pd.DataFrame(task_table(task)) for task in Task}

for task, frame in tables.items():
    display(Markdown(f"### {task.value}"))
    if frame.empty or not frame["n_calls"].any():
        display(Markdown("_Aucune sortie générée pour cette tâche._"))
        continue
    display(frame[COLUMNS].style.format(precision=3).hide(axis="index"))

## Qualité contre latence

Le score composite écrase ces deux axes en un seul chiffre. Les voir séparément montre
s'il existe un modèle qui domine vraiment, ou seulement un compromis.

In [ ]:
import matplotlib.pyplot as plt

usable = {t: f for t, f in tables.items() if not f.empty and f["quality"].notna().any()}

if usable:
    fig, axes = plt.subplots(1, len(usable), figsize=(5 * len(usable), 4.5), squeeze=False)
    for ax, (task, frame) in zip(axes[0], usable.items()):
        ax.scatter(frame["latency_mean_s"], frame["quality"])
        for _, row in frame.iterrows():
            ax.annotate(
                row["model"].split("/")[-1],
                (row["latency_mean_s"], row["quality"]),
                fontsize=7,
                xytext=(4, 4),
                textcoords="offset points",
            )
        ax.set_title(task.value)
        ax.set_xlabel("latence moyenne (s)")
        ax.set_ylabel("qualité [0, 1]")
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Pas encore de résultats d'évaluation à tracer.")

## Meilleure sortie concrète par tâche

Des métriques agrégées ne disent pas si une sortie est utilisable en production.

In [ ]:
for task, frame in tables.items():
    if frame.empty or not frame["n_calls"].any():
        continue
    winner = frame.iloc[0]["model"]
    best = best_output(task, winner)
    if not best:
        continue
    caveat = BENCHMARK_CAVEATS.get(winner)
    display(
        Markdown(
            f"### {task.value} — `{winner}`\n\n"
            f"Document `{best['document_id']}` ({best.get('source_lang')}), "
            f"note du juge : {best.get('judge_score')}\n\n"
            + (f"> Réserve : {caveat}\n\n" if caveat else "")
            + f"```\n{best['output'].strip()[:2500]}\n```"
        )
    )

## Export Markdown

In [ ]:
out = Path.cwd().parent / "reports" / "rapport.md"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(build_report(), encoding="utf-8")
print(f"Rapport écrit dans {out}")